In [ ]:
 %pip install geoai-py

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.9/44.9 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.4/324.4 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 632.9/632.9 kB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.5/122.5 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.7/33.7 MB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 77.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 650.7/650.7 kB 53.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.5/20.5 MB 108.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 102.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.9/243.9 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1

In [ ]:
pip install osmnx rasterstats shapely

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.5/101.5 kB 5.3 MB/s eta 0:00:00


In [ ]:

!rm -rf /content/drive
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir('/content/drive/MyDrive/planner')
print(f"Current working directory: {os.getcwd()}")

import os
print(os.getcwd())        # Shows your current folder
print(os.listdir())       # Lists all files in it

Mounted at /content/drive
Current working directory: /content/drive/MyDrive/planner
/content/drive/MyDrive/planner
['train_segmentation_model.ipynb', 'training_tiles', 'unet_models', 'predictions']


In [ ]:
# =============================================================
# GEOAI URBAN PLANNER – END-TO-END WORKFLOW
# From NAIP images + labels  ➜  Segmentation  ➜  Urban diagnostics
# =============================================================
# Assumptions:
#   - 3-band NAIP imagery (RGB) for training and testing
#   - Vector building footprints for training (GeoJSON)
#   - Semantic classes in prediction raster:
#       1 = buildings, 2 = vegetation, 3 = roads (adjust if different)
# =============================================================

# ---------- 0. INSTALL (ONLY ONCE PER SESSION) ----------
# !pip install -q geoai-py osmnx leafmap rasterstats segmentation-models-pytorch

# ---------- 1. IMPORTS ----------
import os
import geoai
import osmnx as ox
import geopandas as gpd
import rasterio, rasterio.mask
import numpy as np, pandas as pd
from shapely.geometry import box
from rasterstats import zonal_stats
import leafmap
from tempfile import NamedTemporaryFile
from rasterio.features import rasterize
from rasterio.transform import from_bounds
import matplotlib.cm as cm
import matplotlib.colors as colors

print("Using geoai version:", geoai.__version__)
print("Using OSMnx version:", ox.__version__)

# =============================================================
# 2️ PATHS & BASIC CONFIG
# =============================================================

# --- If using your own files in Drive (adjust these paths) ---
BASE_DIR = "/content/drive/MyDrive/planner"

TRAIN_RASTER = os.path.join(BASE_DIR, "naip_rgb_train.tif")
TRAIN_VECTOR = os.path.join(BASE_DIR, "naip_train_buildings.geojson")
TEST_RASTER  = os.path.join(BASE_DIR, "naip_test.tif")


# Folder to store training tiles + model
OUT_FOLDER   = os.path.join(BASE_DIR, "buildings")      # tiles + models
UNET_DIR     = os.path.join(OUT_FOLDER, "unet_models")  # semantic model

# Path where the test prediction (semantic segmentation) will be stored
SEGMENTATION_RASTER = os.path.join(BASE_DIR, "naip_test_semantic_prediction.tif")

# Output diagnostic layers
OUT_GEOJSON_WGS84 = os.path.join(BASE_DIR, "urban_perception_full_wgs84.geojson")

# Grid scale factor – controls analysis resolution (~cell size)
PIX_RES_FACTOR = 100  # each cell ~100× native pixel size (e.g. ≈100m for 1m NAIP)

# =============================================================
# 3️ (OPTIONAL) CREATE TRAINING TILES
#    - Use building vector + training raster to generate tiles
# =============================================================
os.makedirs(OUT_FOLDER, exist_ok=True)

print("Creating training tiles (if not already created)...")
tiles = geoai.export_geotiff_tiles(
    in_raster=TRAIN_RASTER,
    out_folder=OUT_FOLDER,
    in_class_data=TRAIN_VECTOR,
    tile_size=512,
    stride=256,
    buffer_radius=0,
)
print(f"Tiles created: {len(tiles)}")

# =============================================================
# 4️ (OPTIONAL) TRAIN SEMANTIC SEGMENTATION MODEL (U-NET)
#    - You can skip this block if UNET_DIR already contains best_model.pth
# =============================================================
if not os.path.exists(os.path.join(UNET_DIR, "best_model.pth")):
    print("Training U-Net model...")
    geoai.train_segmentation_model(
        images_dir=f"{OUT_FOLDER}/images",
        labels_dir=f"{OUT_FOLDER}/labels",
        output_dir=UNET_DIR,
        architecture="unet",
        encoder_name="resnet34",
        encoder_weights="imagenet",
        num_channels=3,
        num_classes=2,   # background + buildings
        batch_size=8,
        num_epochs=5,    # increase for real training
        learning_rate=0.001,
        val_split=0.2,
        verbose=True,
    )
else:
    print("Found existing U-Net model; skipping training.")

MODEL_PATH = os.path.join(UNET_DIR, "best_model.pth")

# =============================================================
# 5️ RUN SEMANTIC SEGMENTATION ON TEST IMAGE
#    - Produces a 2-class building mask (background/building)
#    - We’ll treat this as the basis for a 3-class raster (bld/veg/road)
#      or extend later with a multi-class model if needed.
# =============================================================
print("Running semantic segmentation on test image...")
geoai.semantic_segmentation(
    input_path=TEST_RASTER,
    output_path=SEGMENTATION_RASTER,
    model_path=MODEL_PATH,
    architecture="unet",
    encoder_name="resnet34",
    num_channels=3,
    num_classes=2,
    window_size=512,
    overlap=256,
    batch_size=4,
)

print(f"Segmentation raster saved to: {SEGMENTATION_RASTER}")

# NOTE:
#  - Currently, the segmentation is binary (building vs background).
#  - For finer separation (vegetation vs roads), you can:
#      * train a multi-class model, OR
#      * combine with NDVI / ancillary data.
#  - For now, we treat:
#      * class 1 = buildings
#      * class 2 = "vegetation proxy" (we’ll compute from NDVI later if desired)


# =============================================================
# 6️ DEFINE ANALYSIS EXTENT (FROM TEST RASTER)
# =============================================================
with rasterio.open(TEST_RASTER) as src:
    bounds = src.bounds
    crs = src.crs
    print(f"Test raster CRS: {crs}")

# Bounding box polygon and WGS84 extent
bbox_geom = box(bounds.left, bounds.bottom, bounds.right, bounds.top)
gdf_bbox = gpd.GeoDataFrame(geometry=[bbox_geom], crs=crs).to_crs(epsg=4326)
lon_min, lat_min, lon_max, lat_max = gdf_bbox.total_bounds
print(f"Study area bbox (lat/lon): {lat_min}, {lat_max}, {lon_min}, {lon_max}")

# =============================================================
# 7️ DOWNLOAD ROAD NETWORK (OSMNX 2.0.6 – CORRECT API)
# =============================================================
from shapely.geometry import Polygon

print("Downloading road network with OSMnx 2.0.6 API...")

polygon = Polygon([
    (lon_min, lat_min),
    (lon_min, lat_max),
    (lon_max, lat_max),
    (lon_max, lat_min),
    (lon_min, lat_min)
])

G = ox.graph_from_polygon(polygon, network_type="drive")

nodes, edges = ox.graph_to_gdfs(G)

print(f"Downloaded {len(edges)} road segments and {len(nodes)} intersections")




# =============================================================
# 8️ CREATE ANALYSIS GRID OVER THE TEST AREA
# =============================================================
with rasterio.open(SEGMENTATION_RASTER) as src:
    seg_bounds, seg_res, seg_crs = src.bounds, src.res[0], src.crs
    step = seg_res * PIX_RES_FACTOR
    xmin, ymin, xmax, ymax = seg_bounds.left, seg_bounds.bottom, seg_bounds.right, seg_bounds.top

grid_cells = [box(x, y, x+step, y+step)
              for x in np.arange(xmin, xmax, step)
              for y in np.arange(ymin, ymax, step)]
grid = gpd.GeoDataFrame(geometry=grid_cells, crs=seg_crs)
print(f"Grid cells created: {len(grid)}")


# =============================================================
#  ADD VEGETATION FROM NDVI (Real Vegetation Detection)
# =============================================================
print("Computing NDVI-based vegetation...")

with rasterio.open(TEST_RASTER) as src:
    red  = src.read(1).astype("float32")
    green = src.read(2).astype("float32")
    blue  = src.read(3).astype("float32")
    nir   = src.read(4).astype("float32")

    ndvi = (nir - red) / (nir + red + 1e-6)  # avoid /0

# Normalize NDVI to a clean 0–1 scale for perception scoring
ndvi_norm = (ndvi - ndvi.min()) / (ndvi.max() - ndvi.min() + 1e-6)

# Create a binary vegetation mask (NDVI threshold)
veg_mask = (ndvi > 0.2).astype("uint8")   # 1 = vegetation, 0 = non-veg

print("NDVI vegetation mask created.")


# =============================================================
# 9️ EXTRACT CLASS RATIOS FROM SEGMENTATION + NDVI VEGETATION
#    - Buildings from segmentation (arr == 1)
#    - Vegetation from NDVI threshold (NDVI > 0.2)
# =============================================================

def compute_ndvi(red, nir):
    """Return NDVI array safely."""
    return (nir - red) / (nir + red + 1e-6)

# Read full test image once (RGB+NIR)
with rasterio.open(TEST_RASTER) as src_full:
    red  = src_full.read(1).astype("float32")
    ner  = src_full.read(3).astype("float32")
    nir  = src_full.read(4).astype("float32")

# List to store stats
stats = []

with rasterio.open(SEGMENTATION_RASTER) as src_seg:
    for geom in grid.geometry:
        try:
            # --- 1: BUILDING COVERAGE FROM SEGMENTATION ---
            seg_arr, _ = rasterio.mask.mask(src_seg, [geom], crop=True)
            seg_arr = seg_arr[0]

            bld_cov = np.count_nonzero(seg_arr == 1) / seg_arr.size

            # --- 2: VEGETATION FROM NDVI ---
            # Clip NAIP (Red + NIR bands)
            out_full, _ = rasterio.mask.mask(src_full, [geom], crop=True)
            r = out_full[0].astype("float32")  # red
            n = out_full[3].astype("float32")  # nir

            ndvi_local = compute_ndvi(r, n)
            veg_cov = np.mean(ndvi_local > 0.2)  # vegetation threshold

            stats.append({
                "bld_coverage": bld_cov,
                "veg_coverage": veg_cov,
            })

        except Exception:
            stats.append({
                "bld_coverage": 0,
                "veg_coverage": 0,
            })

# Attach to grid
grid = pd.concat([grid, pd.DataFrame(stats)], axis=1)


# =============================================================
# 10 COMPUTE ROAD DENSITY (KM / KM²)
# =============================================================
edges_proj = edges.to_crs(grid.crs)
grid["road_density_km_per_km2"] = grid.geometry.apply(
    lambda g: edges_proj.clip(g).length.sum() / 1000.0 / (g.area / 1e6)
)

# =============================================================
# 1️1️ PERCEPTUAL INDICATORS (INSPIRED BY PAPER ON VISUAL SPACE)
#       - greenness, openness, enclosure, walkability, imageability
# =============================================================
def minmax(x):
    return (x - x.min()) / (x.max() - x.min() + 1e-9)

# Greenness proxy: from vegetation coverage (later can be NDVI-based)
grid["greenness"] = minmax(grid["veg_coverage"])

# Openness = 1 – building coverage (sky / openness proxy)
grid["openness"] = 1 - grid["bld_coverage"]

# Enclosure = building coverage (how enclosed the street-space is)
grid["enclosure"] = grid["bld_coverage"]

# Walkability = high road density + low enclosure
grid["walkability"] = (
    minmax(grid["road_density_km_per_km2"]) *
    (1 - minmax(grid["enclosure"]))
).clip(0, 1)

# Imageability proxy: assume unique / intense built form ~ sqrt(coverage)
grid["imageability"] = minmax(np.sqrt(grid["bld_coverage"]))

# =============================================================
# 12 PERCEPTUAL QUALITY INDEX (PQI)
#       - Weighted blend of perception indicators (Paper 1 logic)
# =============================================================
grid["PQI"] = (
    0.25 * grid["greenness"] +
    0.20 * grid["openness"] +
    0.20 * (1 - grid["enclosure"]) +
    0.20 * grid["walkability"] +
    0.15 * grid["imageability"]
).clip(0, 1)

# =============================================================
# 13 URBAN SPRAWL / RISK INDICATORS (BASED ON PAPER 2 LOGIC)
#       - Sprawl, environmental degradation, infrastructure deficiency,
#         and combined risk index
# =============================================================
# Sprawl: low PQI interpreted as more sprawl-like
grid["sprawl_score"] = 1 - grid["PQI"]

# Environmental degradation: low greenness
grid["envdeg_score"] = 1 - grid["greenness"]

# Infrastructure deficiency: low road density & low walkability
grid["infra_deficiency"] = (
    (1 - minmax(grid["road_density_km_per_km2"])) * (1 - grid["walkability"])
)

# Combined risk: weighted blend of the three
grid["combined_risk"] = (
    0.4 * grid["sprawl_score"] +
    0.3 * grid["envdeg_score"] +
    0.3 * grid["infra_deficiency"]
).clip(0, 1)

# =============================================================
# 14 RECOMMENDATION ENGINE
#       - From risk & PQI to planning actions
# =============================================================
def recommend_v2(row):
    recs = []
    if row["combined_risk"] >= 0.6 and row["PQI"] < 0.4:
        recs += [
            "Green corridor & canopy restoration",
            "Retrofit streets for walkability & open sightlines",
            "Add mixed-use nodes to enhance local vitality"
        ]
    elif row["infra_deficiency"] > 0.5:
        recs += [
            "Improve street connectivity and intersection density",
            "Develop pedestrian and cycling infrastructure"
        ]
    elif row["envdeg_score"] > 0.6:
        recs += [
            "Expand urban green infrastructure (parks, linear greenways)",
            "Adopt blue–green stormwater systems"
        ]
    return "; ".join(recs) if recs else "Monitor area; maintain current pattern."

grid["recommendations_v2"] = grid.apply(recommend_v2, axis=1)

# =============================================================
# 15 SUMMARY STATISTICS FOR THE WHOLE IMAGE
# =============================================================
summary = {
    "mean_greenness":      grid["greenness"].mean(),
    "mean_PQI":            grid["PQI"].mean(),
    "mean_combined_risk":  grid["combined_risk"].mean(),
    "high_risk_share":     (grid["combined_risk"] > 0.6).sum() / len(grid),
}
print("\nOverall summary metrics for full image:")
print(pd.DataFrame([summary]))

# =============================================================
# 16 EXPORT GRID AS GEOJSON (WGS84) – MAIN GEOAI PRODUCT
# =============================================================
grid_wgs84 = grid.to_crs(epsg=4326)
grid_wgs84.to_file(OUT_GEOJSON_WGS84, driver="GeoJSON")
print(f"\n Saved diagnostics GeoJSON (WGS84) to: {OUT_GEOJSON_WGS84}")

# =============================================================
# 17 RASTERIZE PQI FOR CLEAN, SCALAR MAP VISUALIZATION
# =============================================================
bounds_w = grid_wgs84.total_bounds
res = 0.0001  # ≈10m at equator; adjust as needed
width = int((bounds_w[2] - bounds_w[0]) / res)
height = int((bounds_w[3] - bounds_w[1]) / res)
transform = from_bounds(*bounds_w, width=width, height=height)

rasterized = rasterize(
    [(geom, val) for geom, val in zip(grid_wgs84.geometry, grid_wgs84["PQI"])],
    out_shape=(height, width),
    transform=transform,
    fill=np.nan,
    dtype="float32"
)

with NamedTemporaryFile(suffix=".tif", delete=False) as tmp:
    PQI_TIF = tmp.name
    with rasterio.open(
        PQI_TIF, "w",
        driver="GTiff",
        height=height,
        width=width,
        count=1,
        dtype=rasterized.dtype,
        crs="EPSG:4326",
        transform=transform,
    ) as dst:
        dst.write(rasterized, 1)

print(f"Raster PQI written to temporary file: {PQI_TIF}")

# =============================================================
#  NEW: RASTERIZE COMBINED RISK (Paper 2 final indicator)
# =============================================================
bounds_w = grid_wgs84.total_bounds
res = 0.0001  # ≈10m at equator; adjust if needed
width = int((bounds_w[2] - bounds_w[0]) / res)
height = int((bounds_w[3] - bounds_w[1]) / res)
transform = from_bounds(*bounds_w, width=width, height=height)

risk_rasterized = rasterize(
    [(geom, val) for geom, val in zip(grid_wgs84.geometry, grid_wgs84["combined_risk"])],
    out_shape=(height, width),
    transform=transform,
    fill=np.nan,
    dtype="float32"
)

with NamedTemporaryFile(suffix=".tif", delete=False) as tmp_risk:
    RISK_TIF = tmp_risk.name
    with rasterio.open(
        RISK_TIF, "w",
        driver="GTiff",
        height=height,
        width=width,
        count=1,
        dtype=risk_rasterized.dtype,
        crs="EPSG:4326",
        transform=transform,
    ) as dst:
        dst.write(risk_rasterized, 1)

print(f"Raster Combined Risk written to temporary file: {RISK_TIF}")


# =============================================================
# 18 VISUALIZATION: AERIAL + PQI + PER-CELL DIAGNOSTICS
# =============================================================
m = leafmap.Map()

# Base imagery
m.add_raster(TEST_RASTER, layer_name="NAIP Base Image")

# Continuous PQI raster
m.add_raster(
    PQI_TIF,
    layer_name="Perceptual Quality Index (PQI)",
    opacity=0.8,
    colormap="YlGn",
)

# Colorbar for PQI
cmap = cm.get_cmap("YlGn")
m.add_colorbar(
    title="Perceptual Quality Index (PQI)",
    colors=[colors.to_hex(cmap(i/10)) for i in range(11)],
    vmin=0,
    vmax=1,
    caption="Low → High Perception Quality",
)


# Combined Risk raster (Paper 2)
m.add_raster(
    RISK_TIF,
    layer_name="Combined Risk (Sprawl–EnvDeg–Infra)",
    opacity=0.8,
    colormap="OrRd",  # Orange to Red = intuitive for risk
)

# Colorbar for Combined Risk
risk_cmap = cm.get_cmap("OrRd")
m.add_colorbar(
    title="Combined Urban Risk",
    colors=[colors.to_hex(risk_cmap(i/10)) for i in range(11)],
    vmin=0,
    vmax=1,
    caption="Low → High Risk",
)

# =============================================================
# SAVE FULL GRID RESULTS TO CSV
# =============================================================
output_csv = "/content/drive/MyDrive/planner/urban_grid_results.csv"

# Convert grid GeoDataFrame to plain DataFrame (remove geometry for CSV)
grid_df = grid.copy()
grid_df["geometry"] = grid_df.geometry.astype(str)

# Save to CSV
grid_df.to_csv(output_csv, index=False)

print(f" Full grid results saved to: {output_csv}")
print(f"Total grid cells exported: {len(grid_df)}")

# Vector diagnostics for hover (risk, recommendations)
m.add_geojson(OUT_GEOJSON_WGS84, layer_name="Perceptual & Risk Diagnostics")

# Zoom to study area
m.zoom_to_bounds(bounds_w.tolist())
m


Using geoai version: 0.18.2
Using OSMnx version: 2.0.7
Creating training tiles (if not already created)...

Raster info for /content/drive/MyDrive/planner/naip_rgb_train.tif:
  CRS: EPSG:26911
  Dimensions: 2503 x 1126
  Resolution: (0.6000000000000046, 0.6)
  Bands: 3
  Bounds: BoundingBox(left=454780.8, bottom=5277567.0, right=456282.6, top=5278242.6)
Loaded 735 features from /content/drive/MyDrive/planner/naip_train_buildings.geojson
Vector CRS: EPSG:4326
Reprojecting features from EPSG:4326 to EPSG:26911
Found 1 unique classes: ['building']


Generated: 36, With features: 36: 100%|██████████| 36/36 [00:10<00:00,  3.43it/s]



------- Export Summary -------
Total tiles exported: 36
Tiles with features: 36 (100.0%)
Average feature pixels per tile: 46795.0
Output saved to: /content/drive/MyDrive/planner/buildings

------- Georeference Verification -------
Tiles created: 5
Found existing U-Net model; skipping training.
Running semantic segmentation on test image...
Input file format: GeoTIFF (.tif)
Processing 65 windows...


84it [00:02, 38.34it/s]


Predicted classes: 2 classes, Background: 95.3%
Inference completed in 2.70 seconds
Saved prediction to /content/drive/MyDrive/planner/naip_test_semantic_prediction.tif
Segmentation raster saved to: /content/drive/MyDrive/planner/naip_test_semantic_prediction.tif
Test raster CRS: EPSG:26911
Study area bbox (lat/lon): 47.64277828744407, 47.650188919438484, -117.6039857669145, -117.57688661080043
Downloaded 303 road segments and 105 intersections
Grid cells created: 476
Computing NDVI-based vegetation...
NDVI vegetation mask created.

Overall summary metrics for full image:
   mean_greenness  mean_PQI  mean_combined_risk  high_risk_share
0             0.0  0.471013            0.666849         0.422269

 Saved diagnostics GeoJSON (WGS84) to: /content/drive/MyDrive/planner/urban_perception_full_wgs84.geojson
Raster PQI written to temporary file: /tmp/tmpk0_vtvid.tif
Raster Combined Risk written to temporary file: /tmp/tmpmu63npyl.tif
 Full grid results saved to: /content/drive/MyDrive/plan

Map(center=[47.646626999999995, -117.59036599999999], controls=(ZoomControl(options=['position', 'zoom_in_text…

In [ ]:
import rasterio

naip_path = "/content/drive/MyDrive/planner/naip_test.tif"

with rasterio.open(naip_path) as src:
    print("Number of bands:", src.count)
    print("CRS:", src.crs)
    print("Resolution:", src.res)
    print("Bounds:", src.bounds)
    print("Band descriptions:", src.descriptions)


Number of bands: 4
CRS: EPSG:26911
Resolution: (0.6000000000000034, 0.5999999999994469)
Bounds: BoundingBox(left=454641.6, bottom=5276774.4, right=456670.8, top=5277582.6)
Band descriptions: (None, None, None, None)


MULTI-STATE IMPLEMENTATION

In [ ]:
 %pip install geoai-py

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.9/44.9 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 306.0/306.0 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 632.9/632.9 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.5/122.5 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.7/33.7 MB 39.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 51.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 650.7/650.7 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.5/20.5 MB 74.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 60.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.9/243.9 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1

In [ ]:
pip install osmnx rasterstats shapely

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.5/101.5 kB 4.5 MB/s eta 0:00:00


In [ ]:
# =============================================================
# STEP 1 — DATA ACCESS, EXTRACTION, AND STATE DISCOVERY
# =============================================================
# Purpose:
#   - Mount Google Drive
#   - Unzip NAIP and building datasets
#   - Identify U.S. states with BOTH imagery and buildings
#   - Define the valid state list for downstream GeoAI processing
# =============================================================

# ---------- 1.1 Mount Google Drive ----------
from google.colab import drive
drive.mount('/content/drive')

import os
import zipfile

BASE_DIR = "/content/drive/MyDrive"
os.chdir(BASE_DIR)

print(f" Working directory: {os.getcwd()}")
print(" Available files:", os.listdir())

# ---------- 1.2 Define ZIP paths ----------
NAIP_ZIP = os.path.join(BASE_DIR, "NAIP_50_STATES.zip")
BLD_ZIP  = os.path.join(BASE_DIR, "BUILDINGS_50_STATES.zip")

NAIP_DIR = os.path.join(BASE_DIR, "NAIP_50_STATES")
BLD_DIR  = os.path.join(BASE_DIR, "BUILDINGS_50_STATES")

# ---------- 1.3 Unzip datasets if not already extracted ----------
def unzip_if_needed(zip_path, out_dir):
    if not os.path.exists(out_dir):
        print(f" Extracting {os.path.basename(zip_path)} ...")
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(out_dir)
        print(f" Extracted to {out_dir}")
    else:
        print(f"✔ {out_dir} already exists — skipping unzip")

unzip_if_needed(NAIP_ZIP, NAIP_DIR)
unzip_if_needed(BLD_ZIP, BLD_DIR)

# ---------- 1.4 Discover valid states ----------
naip_states = {
    s for s in os.listdir(NAIP_DIR)
    if os.path.isdir(os.path.join(NAIP_DIR, s))
}

bld_states = {
    s for s in os.listdir(BLD_DIR)
    if os.path.isdir(os.path.join(BLD_DIR, s))
}

valid_states = sorted(list(naip_states.intersection(bld_states)))

print("\n States with BOTH NAIP imagery and building footprints:")
print(f"Total valid states: {len(valid_states)}")
print(valid_states)

# ---------- 1.5 Persist state list for pipeline ----------
VALID_STATES = valid_states

Mounted at /content/drive
 Working directory: /content/drive/MyDrive
 Available files: ['Essay on Program_Africa Scholarship Program.docx', 'IDEA CONSULT GH', 'Flood Gov. Aglago', 'SP', 'Legon District', 'ST. CECILIA SCH', 'SP2_Continuous Assessment-sem 2.docx', "Dev't Plg & Management.zip", 'Phd.zip', 'MPhil Urban Management.zip', 'IDEA_Proposal_Structure.pdf', 'IDEA-Spatial Ed-Meeting.docx', 'AdmissionDocument.pdf', 'App letter - Regina.docx', 'Desmond_EnDev_Indepth-study_Somanya.docx', 'SCAN DOC', 'MOFA_Somanya.3gpp', '1st Year', 'GEOGRAPHY FOR PLANNERS. PASSCO.. AIBI ANDREWS.pdf', 'PL 159_Semester One Notes_Econs.doc', 'Lesson4- MAP REPRODUCTION.pdf', 'ICT for the planners 2014-2015- email.pdf', '[J._Crawshaw,_J._Chambers]_A_Concise_Course_in_Adv(BookFi.org).pdf', 'Planning Laws', 'Friends Projects', 'Group Activity Matrix', 'Brief for Assignment 1.docx', 'waste-management.pdf', 'Group 6 _SEM 2 _Works', 'Contract (English).pdf', 'DAglago_Final Draft.docx', 'Paper Structure.docx', '

In [ ]:
# =============================================================
# 1️ DISCOVER VALID STATES (NAIP + BUILDINGS AVAILABLE)
# =============================================================

import os

BASE_DIR = "/content/drive/MyDrive"

NAIP_DIR = os.path.join(BASE_DIR, "NAIP_50_STATES")
BLD_DIR  = os.path.join(BASE_DIR, "BUILDINGS_50_STATES")

naip_files = os.listdir(NAIP_DIR)
bld_files  = os.listdir(BLD_DIR)

naip_states = {
    f.replace("NAIP_", "").replace(".tif", "")
    for f in naip_files
    if f.startswith("NAIP_") and f.endswith(".tif")
}

bld_states = {
    f.replace("buildings_", "").replace(".geojson", "")
    for f in bld_files
    if f.startswith("buildings_") and f.endswith(".geojson")
}

valid_states = sorted(list(naip_states.intersection(bld_states)))

print(f" States with BOTH datasets: {len(valid_states)}")
print(valid_states)

# 🔹 LIMIT TO FIRST 5 (SAFE TEST RUN)
TEST_STATES = valid_states[:5]
print("\n Processing ONLY these states for now:")
print(TEST_STATES)


 States with BOTH datasets: 47
['AL', 'AR', 'AZ', 'CA', 'CO', 'CT', 'DE', 'FL', 'GA', 'IA', 'ID', 'IL', 'IN', 'KS', 'KY', 'LA', 'MA', 'MD', 'ME', 'MI', 'MN', 'MO', 'MS', 'MT', 'NC', 'ND', 'NE', 'NH', 'NJ', 'NM', 'NV', 'NY', 'OH', 'OK', 'PA', 'RI', 'SC', 'SD', 'TN', 'TX', 'UT', 'VA', 'VT', 'WA', 'WI', 'WV', 'WY']

 Processing ONLY these states for now:
['AL', 'AR', 'AZ', 'CA', 'CO']


In [ ]:
# =============================================================
# 2️ TRAINING TILE OUTPUT STRUCTURE
# =============================================================

PLANNER_DIR = os.path.join(BASE_DIR, "planner")
TRAIN_TILES_ROOT = os.path.join(PLANNER_DIR, "training_tiles")

os.makedirs(TRAIN_TILES_ROOT, exist_ok=True)

print(f" Training tiles root: {TRAIN_TILES_ROOT}")


 Training tiles root: /content/drive/MyDrive/planner/training_tiles


In [ ]:
# =============================================================
# 3️ CREATE TRAINING TILES (SAFE, STATE-BY-STATE)
# =============================================================

import geoai
import gc

TRAIN_TILES_ROOT = os.path.join(BASE_DIR, "planner", "training_tiles")
os.makedirs(TRAIN_TILES_ROOT, exist_ok=True)

all_tiles = []

print("\n Creating training tiles across states...\n")

for state in TEST_STATES:
    try:
        print(f"\n→ Sampling tiles from {state}")

        raster_path = os.path.join(NAIP_DIR, f"NAIP_{state}.tif")
        vector_path = os.path.join(BLD_DIR, f"buildings_{state}.geojson")

        out_dir = os.path.join(TRAIN_TILES_ROOT, state)
        os.makedirs(out_dir, exist_ok=True)

        tiles = geoai.export_geotiff_tiles(
            in_raster=raster_path,
            out_folder=out_dir,
            in_class_data=vector_path,
            tile_size=512,
            stride=256,
            buffer_radius=0,
        )

        print(f"✓ {state}: {len(tiles)} tiles created")
        all_tiles.extend(tiles)

        # 🔹 CRITICAL: free memory after each state
        del tiles
        gc.collect()

    except Exception as e:
        print(f" Skipping {state} due to error: {e}")



🧩 Creating training tiles across states...


→ Sampling tiles from AL

Raster info for /content/drive/MyDrive/NAIP_50_STATES/NAIP_AL.tif:
  CRS: EPSG:4326
  Dimensions: 7589 x 4009
  Resolution: (5.38989170471713e-06, 5.38989170471713e-06)
  Bands: 3
  Bounds: BoundingBox(left=-86.85295208504682, bottom=33.51419570296337, right=-86.81204819689972, top=33.53580377880758)
Loaded 3184 features from /content/drive/MyDrive/BUILDINGS_50_STATES/buildings_AL.geojson
Vector CRS: EPSG:4326


Generated: 415, With features: 410:  95%|█████████▌| 415/435 [05:00<00:16,  1.20it/s]

In [ ]:
GLOBAL_IMG_DIR = os.path.join(TRAIN_TILES_ROOT, "images")
GLOBAL_LBL_DIR = os.path.join(TRAIN_TILES_ROOT, "labels")

os.makedirs(GLOBAL_IMG_DIR, exist_ok=True)
os.makedirs(GLOBAL_LBL_DIR, exist_ok=True)


In [ ]:
# =============================================================
# 4
# =============================================================
import shutil

print("\n Merging tiles into global training folders...\n")

for state in TEST_STATES:
    state_dir = os.path.join(TRAIN_TILES_ROOT, state)

    img_dir = os.path.join(state_dir, "images")
    lbl_dir = os.path.join(state_dir, "labels")

    if not os.path.exists(img_dir) or not os.path.exists(lbl_dir):
        print(f" Skipping {state}: missing images/labels")
        continue

    for f in os.listdir(img_dir):
        shutil.copy(os.path.join(img_dir, f),
                    os.path.join(GLOBAL_IMG_DIR, f"{state}_{f}"))

    for f in os.listdir(lbl_dir):
        shutil.copy(os.path.join(lbl_dir, f),
                    os.path.join(GLOBAL_LBL_DIR, f"{state}_{f}"))

print(" Global training dataset ready")
print("Images:", len(os.listdir(GLOBAL_IMG_DIR)))
print("Labels:", len(os.listdir(GLOBAL_LBL_DIR)))



📦 Merging tiles into global training folders...

✅ Global training dataset ready
Images: 2175
Labels: 2175


In [ ]:
# =============================================================
# 4a
# =============================================================

import geoai
UNET_DIR = os.path.join(BASE_DIR, "planner", "unet_models")
os.makedirs(UNET_DIR, exist_ok=True)

print(" Training global U-Net model (5 states)...")

geoai.train_segmentation_model(
    images_dir=GLOBAL_IMG_DIR,
    labels_dir=GLOBAL_LBL_DIR,
    output_dir=UNET_DIR,
    architecture="unet",
    encoder_name="resnet34",
    encoder_weights="imagenet",
    num_channels=3,
    num_classes=2,      # background + buildings
    batch_size=8,
    num_epochs=1,       # keep low for now
    learning_rate=0.001,
    val_split=0.2,
    verbose=True,
)


🧠 Training global U-Net model (5 states)...
Using device: cpu
Found 2175 image files and 2175 label files
Training on 1740 images, validating on 435 images
Checking image sizes for compatibility...
All sampled images have the same size: (512, 512)
No resizing needed.
Testing data loader...
Data loader test passed.


config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/87.3M [00:00<?, ?B/s]

Starting training with unet + resnet34
Model parameters: 24,436,514
Epoch: 1, Batch: 1/218, Loss: 0.5952, Time: 136.18s
Epoch: 1, Batch: 11/218, Loss: 0.5871, Time: 478.68s
Epoch: 1, Batch: 21/218, Loss: 0.5031, Time: 453.25s
Epoch: 1, Batch: 31/218, Loss: 0.5118, Time: 460.96s
Epoch: 1, Batch: 41/218, Loss: 0.4165, Time: 447.50s
Epoch: 1, Batch: 51/218, Loss: 0.3557, Time: 449.42s
Epoch: 1, Batch: 61/218, Loss: 0.4846, Time: 444.90s
Epoch: 1, Batch: 71/218, Loss: 0.3486, Time: 456.88s
Epoch: 1, Batch: 81/218, Loss: 0.3948, Time: 441.01s
Epoch: 1, Batch: 91/218, Loss: 0.3461, Time: 474.91s
Epoch: 1, Batch: 101/218, Loss: 0.4323, Time: 454.76s
Epoch: 1, Batch: 111/218, Loss: 0.2818, Time: 444.76s
Epoch: 1, Batch: 121/218, Loss: 0.2911, Time: 448.79s
Epoch: 1, Batch: 131/218, Loss: 0.2812, Time: 442.95s
Epoch: 1, Batch: 141/218, Loss: 0.3197, Time: 456.98s
Epoch: 1, Batch: 151/218, Loss: 0.3253, Time: 448.52s
Epoch: 1, Batch: 161/218, Loss: 0.3116, Time: 447.37s
Epoch: 1, Batch: 171/218,

In [ ]:
# =============================================================
# 5️ SEMANTIC SEGMENTATION (INFERENCE – MULTI-STATE)
# =============================================================

import os
import geoai
import gc

MODEL_PATH = os.path.join(
    BASE_DIR,
    "planner",
    "unet_models",
    "best_model.pth"
)

PRED_DIR = os.path.join(BASE_DIR, "planner", "predictions")
os.makedirs(PRED_DIR, exist_ok=True)

print(" Running inference using trained global model...")


🧠 Running inference using trained global model...


In [ ]:
for state in TEST_STATES:
    try:
        print(f"\n Running inference for {state}")

        input_raster = os.path.join(NAIP_DIR, f"NAIP_{state}.tif")
        output_raster = os.path.join(PRED_DIR, f"{state}_buildings_pred.tif")

        geoai.semantic_segmentation(
            input_path=input_raster,
            output_path=output_raster,
            model_path=MODEL_PATH,
            architecture="unet",
            encoder_name="resnet34",
            num_channels=3,
            num_classes=2,
            window_size=512,
            overlap=256,
            batch_size=2,   # CPU-safe
        )

        print(f" Saved prediction: {output_raster}")

        gc.collect()

    except Exception as e:
        print(f" Skipping {state} due to error: {e}")



🔍 Running inference for AL
Input file format: GeoTIFF (.tif)
Processing 435 windows...


480it [12:41,  1.59s/it]


Predicted classes: 2 classes, Background: 79.6%
Inference completed in 766.04 seconds
Saved prediction to /content/drive/MyDrive/planner/predictions/AL_buildings_pred.tif
✅ Saved prediction: /content/drive/MyDrive/planner/predictions/AL_buildings_pred.tif

🔍 Running inference for AR
Input file format: GeoTIFF (.tif)
Processing 435 windows...


480it [12:57,  1.62s/it]


Predicted classes: 2 classes, Background: 75.3%
Inference completed in 782.73 seconds
Saved prediction to /content/drive/MyDrive/planner/predictions/AR_buildings_pred.tif
✅ Saved prediction: /content/drive/MyDrive/planner/predictions/AR_buildings_pred.tif

🔍 Running inference for AZ
Input file format: GeoTIFF (.tif)
Processing 435 windows...


...

Predicted classes: 2 classes, Background: 79.6%
Inference completed in 768.84 seconds
Saved prediction to /content/drive/MyDrive/planner/predictions/AZ_buildings_pred.tif
✅ Saved prediction: /content/drive/MyDrive/planner/predictions/AZ_buildings_pred.tif

🔍 Running inference for CA
Input file format: GeoTIFF (.tif)
Processing 435 windows...


480it [13:05,  1.64s/it]


Predicted classes: 2 classes, Background: 44.8%
Inference completed in 791.96 seconds
Saved prediction to /content/drive/MyDrive/planner/predictions/CA_buildings_pred.tif
✅ Saved prediction: /content/drive/MyDrive/planner/predictions/CA_buildings_pred.tif

🔍 Running inference for CO
Input file format: GeoTIFF (.tif)
Processing 435 windows...


480it [12:54,  1.61s/it]


Predicted classes: 2 classes, Background: 67.1%
Inference completed in 779.30 seconds
Saved prediction to /content/drive/MyDrive/planner/predictions/CO_buildings_pred.tif
✅ Saved prediction: /content/drive/MyDrive/planner/predictions/CO_buildings_pred.tif


In [ ]:
import rasterio
import rasterio.mask
import geopandas as gpd
import numpy as np
import pandas as pd
import os, gc
from shapely.geometry import box, Polygon
import osmnx as ox


PRED_DIR = "/content/drive/MyDrive/planner/predictions"
NAIP_DIR = "/content/drive/MyDrive/NAIP_50_STATES"
OUT_ANALYSIS = "/content/drive/MyDrive/planner/analysis_outputs"
os.makedirs(OUT_ANALYSIS, exist_ok=True)

PIX_RES_FACTOR = 100



In [ ]:
for state in TEST_STATES:
    print(f"\n Processing analysis for {state}")

    try:
        TEST_RASTER = os.path.join(NAIP_DIR, f"NAIP_{state}.tif")
        SEGMENTATION_RASTER = os.path.join(PRED_DIR, f"{state}_buildings_pred.tif")

        # =============================================================
        # 6️ DEFINE ANALYSIS EXTENT
        # =============================================================
        with rasterio.open(TEST_RASTER) as src:
            bounds = src.bounds
            crs = src.crs

        bbox_geom = box(bounds.left, bounds.bottom, bounds.right, bounds.top)
        gdf_bbox = gpd.GeoDataFrame(geometry=[bbox_geom], crs=crs).to_crs(epsg=4326)
        lon_min, lat_min, lon_max, lat_max = gdf_bbox.total_bounds

        # =============================================================
        # 7️ ROAD NETWORK (STATE CLIPPED)
        # =============================================================
        polygon = Polygon([
            (lon_min, lat_min),
            (lon_min, lat_max),
            (lon_max, lat_max),
            (lon_max, lat_min),
            (lon_min, lat_min)
        ])

        G = ox.graph_from_polygon(polygon, network_type="drive")
        nodes, edges = ox.graph_to_gdfs(G)

        # =============================================================
        # 8️ CREATE GRID
        # =============================================================
        with rasterio.open(SEGMENTATION_RASTER) as src:
            seg_bounds = src.bounds
            seg_res = src.res[0]
            seg_crs = src.crs

        step = seg_res * PIX_RES_FACTOR
        xmin, ymin, xmax, ymax = seg_bounds

        grid_cells = [
            box(x, y, x + step, y + step)
            for x in np.arange(xmin, xmax, step)
            for y in np.arange(ymin, ymax, step)
        ]

        grid = gpd.GeoDataFrame(geometry=grid_cells, crs=seg_crs)

        # =============================================================
        # 9️ BUILDING + VEGETATION COVERAGE
        # =============================================================
        def compute_ndvi(r, n):
            return (n - r) / (n + r + 1e-6)

        stats = []

        with rasterio.open(SEGMENTATION_RASTER) as src_seg, \
             rasterio.open(TEST_RASTER) as src_full:

            for geom in grid.geometry:
                try:
                    seg_arr, _ = rasterio.mask.mask(src_seg, [geom], crop=True)
                    seg_arr = seg_arr[0]
                    bld_cov = np.count_nonzero(seg_arr == 1) / seg_arr.size

                    out_full, _ = rasterio.mask.mask(src_full, [geom], crop=True)
                    red = out_full[0].astype("float32")
                    nir = out_full[3].astype("float32")

                    ndvi = compute_ndvi(red, nir)
                    veg_cov = np.mean(ndvi > 0.2)

                    stats.append({
                        "bld_coverage": bld_cov,
                        "veg_coverage": veg_cov,
                    })

                except Exception:
                    stats.append({
                        "bld_coverage": 0,
                        "veg_coverage": 0,
                    })

        grid = pd.concat([grid, pd.DataFrame(stats)], axis=1)

        # Save per-state output
        out_file = os.path.join(OUT_ANALYSIS, f"{state}_grid.geojson")
        grid.to_crs(epsg=4326).to_file(out_file, driver="GeoJSON")

        print(f" Saved grid for {state}: {out_file}")

        # CLEANUP
        del grid, stats, edges, nodes
        gc.collect()

    except Exception as e:
        print(f" Skipping {state}: {e}")



📍 Processing analysis for AL
✅ Saved grid for AL: /content/drive/MyDrive/planner/analysis_outputs/AL_grid.geojson

📍 Processing analysis for AR
✅ Saved grid for AR: /content/drive/MyDrive/planner/analysis_outputs/AR_grid.geojson

📍 Processing analysis for AZ
✅ Saved grid for AZ: /content/drive/MyDrive/planner/analysis_outputs/AZ_grid.geojson

📍 Processing analysis for CA
✅ Saved grid for CA: /content/drive/MyDrive/planner/analysis_outputs/CA_grid.geojson

📍 Processing analysis for CO
✅ Saved grid for CO: /content/drive/MyDrive/planner/analysis_outputs/CO_grid.geojson


In [ ]:
# =============================================================
# 10–18 URBAN DIAGNOSTICS + EXPORTS + ONE-STATE VISUALIZATION
# (Robust version: fixes missing `grid`, missing `edges`, missing map display)
# =============================================================
# What this block assumes you ALREADY have:
#   - BASE_DIR, NAIP_DIR, OUT_ANALYSIS exist (from your earlier steps)
#   - You have saved: /planner/analysis_outputs/{STATE}_grid.geojson  (from step 6–9 loop)
#   - You want to pick ONE state to visualize (VIS_STATE), but still be able to run diagnostics for any state

import os, gc
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Polygon
import osmnx as ox

from rasterio.features import rasterize
from rasterio.transform import from_bounds
import rasterio

import leafmap
import matplotlib.cm as cm
import matplotlib.colors as colors

# -----------------------------
# CONFIG: pick your state here
# -----------------------------
VIS_STATE = "CO"   # change anytime: "AL","AR","AZ","CA","CO"
BASE_DIR  = "/content/drive/MyDrive"   # keep consistent with your notebook

NAIP_DIR = os.path.join(BASE_DIR, "NAIP_50_STATES")

# Output folders (consistent + reusable)
OUT_ANALYSIS = os.path.join(BASE_DIR, "planner", "analysis_outputs")   # grid + diagnostics geojson
OUT_RASTERS  = os.path.join(BASE_DIR, "planner", "analysis_rasters")   # PQI + risk tifs
OUT_CSV      = os.path.join(BASE_DIR, "planner", "analysis_csv")       # CSV summaries

os.makedirs(OUT_ANALYSIS, exist_ok=True)
os.makedirs(OUT_RASTERS, exist_ok=True)
os.makedirs(OUT_CSV, exist_ok=True)

# ---------------------------------------
# Helpers
# ---------------------------------------
def minmax(x: pd.Series) -> pd.Series:
    x = x.astype(float)
    return (x - x.min()) / (x.max() - x.min() + 1e-9)

def rasterize_grid(grid_wgs84: gpd.GeoDataFrame, value_col: str, out_tif_path: str, res=0.0001):
    """
    Rasterize a grid GeoDataFrame to GeoTIFF (EPSG:4326).
    res=0.0001 ~ 10m-ish at equator.
    Returns (bounds) to zoom map.
    """
    bounds = grid_wgs84.total_bounds  # (minx, miny, maxx, maxy)
    width  = int((bounds[2] - bounds[0]) / res)
    height = int((bounds[3] - bounds[1]) / res)

    if width <= 0 or height <= 0:
        raise ValueError("Invalid raster dimensions from bounds/res.")

    transform = from_bounds(*bounds, width=width, height=height)

    arr = rasterize(
        [(geom, float(val)) for geom, val in zip(grid_wgs84.geometry, grid_wgs84[value_col])],
        out_shape=(height, width),
        transform=transform,
        fill=np.nan,
        dtype="float32"
    )

    with rasterio.open(
        out_tif_path, "w",
        driver="GTiff",
        height=height,
        width=width,
        count=1,
        dtype="float32",
        crs="EPSG:4326",
        transform=transform,
    ) as dst:
        dst.write(arr, 1)

    return bounds

def utm_epsg_from_lonlat(lon: float, lat: float) -> int:
    utm_zone = int((lon + 180) / 6) + 1
    return (32600 + utm_zone) if lat >= 0 else (32700 + utm_zone)

def build_osm_edges_for_grid_bbox(grid_wgs84: gpd.GeoDataFrame):
    """
    Rebuild OSM road network for EXACT grid bbox (not a random +/-0.05 box).
    This prevents missing edges + inconsistent density.
    """
    minx, miny, maxx, maxy = grid_wgs84.total_bounds

    poly = Polygon([
        (minx, miny),
        (minx, maxy),
        (maxx, maxy),
        (maxx, miny),
        (minx, miny)
    ])

    G = ox.graph_from_polygon(poly, network_type="drive")
    _, edges = ox.graph_to_gdfs(G)
    return edges

def show_leafmap(TEST_RASTER, PQI_TIF, RISK_TIF, geojson_path, bounds_w):
    """
    Leafmap sometimes won't render if the map object isn't the last line in a cell.
    This function returns `m` — make sure you put `m` alone on the last line.
    """
    m = leafmap.Map()

    m.add_raster(TEST_RASTER, layer_name="NAIP")

    m.add_raster(PQI_TIF, layer_name="PQI", colormap="YlGn", opacity=0.8)
    cmap = cm.get_cmap("YlGn")
    m.add_colorbar(
        title="PQI",
        colors=[colors.to_hex(cmap(i/10)) for i in range(11)],
        vmin=0, vmax=1,
        caption="Low → High"
    )

    m.add_raster(RISK_TIF, layer_name="Combined Risk", colormap="OrRd", opacity=0.8)
    rcmap = cm.get_cmap("OrRd")
    m.add_colorbar(
        title="Combined Risk",
        colors=[colors.to_hex(rcmap(i/10)) for i in range(11)],
        vmin=0, vmax=1,
        caption="Low → High"
    )

    m.add_geojson(geojson_path, layer_name="Diagnostics (grid)")
    m.zoom_to_bounds(list(bounds_w))
    return m

# ---------------------------------------
# MAIN: Run diagnostics for one state
# ---------------------------------------
state = VIS_STATE

GRID_FILE = os.path.join(OUT_ANALYSIS, f"{state}_grid.geojson")
TEST_RASTER = os.path.join(NAIP_DIR, f"NAIP_{state}.tif")

if not os.path.exists(GRID_FILE):
    raise RuntimeError(f"{state}: Missing grid file. Expected: {GRID_FILE}")

if not os.path.exists(TEST_RASTER):
    raise RuntimeError(f"{state}: Missing NAIP raster. Expected: {TEST_RASTER}")

grid = gpd.read_file(GRID_FILE)
if grid.empty:
    raise RuntimeError(f"{state}: Grid is empty in {GRID_FILE}")

print(f" Grid loaded for {state}: {len(grid)} cells")

# Ensure required columns exist (from step 9)
required_cols = ["bld_coverage", "veg_coverage"]
missing = [c for c in required_cols if c not in grid.columns]
if missing:
    raise RuntimeError(f"{state}: Grid is missing required columns: {missing}. Re-run step 6–9 to rebuild grid.")

# Make sure grid CRS exists
if grid.crs is None:
    raise RuntimeError(f"{state}: Grid CRS is None. Recreate grid with a valid CRS before diagnostics.")

# ---- Recreate roads properly from grid bbox (fixes 'edges not defined') ----
grid_wgs84_tmp = grid.to_crs(epsg=4326)
edges = build_osm_edges_for_grid_bbox(grid_wgs84_tmp)
print(f"🛣️ Roads loaded for {state}: {len(edges)} segments")

# =============================================================
# 10) ROAD DENSITY (KM / KM²) — METRIC CRS (UTM)
# =============================================================
centroid = grid_wgs84_tmp.unary_union.centroid
lonc, latc = centroid.x, centroid.y
epsg_utm = utm_epsg_from_lonlat(lonc, latc)

grid_m  = grid.to_crs(epsg=epsg_utm)
edges_m = edges.to_crs(epsg=epsg_utm)

# Clip-based road density per cell (safe)
def road_density(cell_geom):
    # clip() can be slow; but stable. You can optimize later if needed.
    clipped = edges_m.clip(cell_geom)
    road_km = clipped.length.sum() / 1000.0
    area_km2 = cell_geom.area / 1e6
    return road_km / area_km2 if area_km2 > 0 else 0.0

grid_m["road_density_km_per_km2"] = grid_m.geometry.apply(road_density)
grid["road_density_km_per_km2"] = grid_m["road_density_km_per_km2"].values

# =============================================================
# 11) PERCEPTUAL INDICATORS
# =============================================================
grid["greenness"] = minmax(grid["veg_coverage"])
grid["openness"]  = (1 - grid["bld_coverage"]).clip(0, 1)
grid["enclosure"] = grid["bld_coverage"].clip(0, 1)

grid["walkability"] = (
    minmax(grid["road_density_km_per_km2"]) *
    (1 - minmax(grid["enclosure"]))
).clip(0, 1)

grid["imageability"] = minmax(np.sqrt(grid["bld_coverage"].clip(0, 1)))

# =============================================================
# 12) PQI
# =============================================================
grid["PQI"] = (
    0.25 * grid["greenness"] +
    0.20 * grid["openness"] +
    0.20 * (1 - grid["enclosure"]) +
    0.20 * grid["walkability"] +
    0.15 * grid["imageability"]
).clip(0, 1)

# =============================================================
# 13) RISK INDICATORS
# =============================================================
grid["sprawl_score"] = (1 - grid["PQI"]).clip(0, 1)
grid["envdeg_score"] = (1 - grid["greenness"]).clip(0, 1)

grid["infra_deficiency"] = (
    (1 - minmax(grid["road_density_km_per_km2"])) *
    (1 - grid["walkability"])
).clip(0, 1)

grid["combined_risk"] = (
    0.4 * grid["sprawl_score"] +
    0.3 * grid["envdeg_score"] +
    0.3 * grid["infra_deficiency"]
).clip(0, 1)

# =============================================================
# 14) RECOMMENDATIONS
# =============================================================
def recommend(row):
    if row["combined_risk"] >= 0.6 and row["PQI"] < 0.4:
        return "Green corridors; Walkability retrofit; Mixed-use densification"
    if row["infra_deficiency"] > 0.5:
        return "Improve connectivity; Pedestrian & cycling infrastructure"
    if row["envdeg_score"] > 0.6:
        return "Urban greening; Blue–green infrastructure"
    return "Maintain & monitor"

grid["recommendations"] = grid.apply(recommend, axis=1)

# =============================================================
# 15) SUMMARY
# =============================================================
summary = pd.DataFrame([{
    "state": state,
    "mean_PQI": float(grid["PQI"].mean()),
    "mean_combined_risk": float(grid["combined_risk"].mean()),
    "high_risk_share": float((grid["combined_risk"] > 0.6).mean())
}])
print(summary)

# =============================================================
# 16) EXPORT VECTOR OUTPUTS
# =============================================================
grid_wgs84 = grid.to_crs(epsg=4326)

geojson_path = os.path.join(OUT_ANALYSIS, f"{state}_urban_diagnostics.geojson")
csv_path     = os.path.join(OUT_CSV, f"{state}_urban_grid_results.csv")

grid_wgs84.to_file(geojson_path, driver="GeoJSON")

df = grid_wgs84.copy()
df["geometry"] = df.geometry.astype(str)
df.to_csv(csv_path, index=False)

print(f" {state}: GeoJSON exported -> {geojson_path}")
print(f" {state}: CSV exported    -> {csv_path}")

# =============================================================
# 17) RASTERIZE PQI + RISK
# =============================================================
PQI_TIF  = os.path.join(OUT_RASTERS, f"{state}_PQI.tif")
RISK_TIF = os.path.join(OUT_RASTERS, f"{state}_combined_risk.tif")

bounds_w = rasterize_grid(grid_wgs84, "PQI", PQI_TIF, res=0.0001)
_        = rasterize_grid(grid_wgs84, "combined_risk", RISK_TIF, res=0.0001)

print(f" {state}: PQI raster  -> {PQI_TIF}")
print(f" {state}: Risk raster -> {RISK_TIF}")

# =============================================================
# 18) MAP (IMPORTANT: return map object and display it)
# =============================================================
m = show_leafmap(TEST_RASTER, PQI_TIF, RISK_TIF, geojson_path, bounds_w)

# Cleanup (optional) -----
del grid_m, edges_m
gc.collect()

# IMPORTANT:

m




 Grid loaded for CO: 3116 cells
🛣️ Roads loaded for CO: 1789 segments
  state  mean_PQI  mean_combined_risk  high_risk_share
0    CO  0.443774            0.720292         0.857831
 CO: GeoJSON exported -> /content/drive/MyDrive/planner/analysis_outputs/CO_urban_diagnostics.geojson
 CO: CSV exported    -> /content/drive/MyDrive/planner/analysis_csv/CO_urban_grid_results.csv
 CO: PQI raster  -> /content/drive/MyDrive/planner/analysis_rasters/CO_PQI.tif
 CO: Risk raster -> /content/drive/MyDrive/planner/analysis_rasters/CO_combined_risk.tif


Map(center=[39.7452445, -104.98997], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title…